In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, collect_set, concat_ws, row_number, lit, trim
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

# 1. SETUP
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"
builder = SparkSession.builder \
    .appName("Gold_Dim_Filme_Final_Inteiros") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# 2. CAMINHOS (Confirma se estão certos)
paths = {
    "Netflix": "hdfs://hdfs-nn:9000/warehouse/silver.db/netflix_titles",
    "Amazon":  "hdfs://hdfs-nn:9000/warehouse/silver.db/titles",
    "Movies":  "hdfs://hdfs-nn:9000/warehouse/silver.db/movies_silver",
    "BoxOffice": "hdfs://hdfs-nn:9000/warehouse/silver.db/boxoffice",
    "Rotten":  "hdfs://hdfs-nn:9000/warehouse/silver.db/rotten_tomatoes"
}

# 3. LÓGICA DE LEITURA
def ler_filmes(source_name, path):
    try:
        df = spark.read.format("delta").load(path)
        cols = df.columns
        c_title = "movie_title" if "movie_title" in cols else "title"
        c_genre = "genres" if "genres" in cols else ("genre" if "genre" in cols else None)
        
        df = df.withColumnRenamed(c_title, "title")
        if c_genre:
            df = df.withColumnRenamed(c_genre, "genero")
        else:
            df = df.withColumn("genero", lit(None).cast("string"))
            
        return df.select(trim(col("title")).alias("title"), col("genero"))
    except Exception as e:
        print(f"⚠️ Saltado: {source_name}")
        return None

dfs_filmes = []
for nome, caminho in paths.items():
    df_temp = ler_filmes(nome, caminho)
    if df_temp: dfs_filmes.append(df_temp)

if dfs_filmes:
    df_uniao = dfs_filmes[0]
    for d in dfs_filmes[1:]: 
        df_uniao = df_uniao.union(d)

    # ===================================================
    # 4. AQUI ESTÁ A MUDANÇA (IDs Inteiros)
    # ===================================================
    print("🧹 A criar IDs numéricos...")
    
    df_gold_filmes = df_uniao \
        .groupBy("title") \
        .agg(concat_ws(", ", collect_set("genero")).alias("genero")) \
        .filter(col("title") != "")
    
    window_spec = Window.orderBy("title")
    
    # REMOVI O CONCAT("F", ...). FICA SÓ O NÚMERO.
    df_dim_filme = df_gold_filmes.withColumn("id_filme", row_number().over(window_spec)) \
        .select("id_filme", "title", "genero")

    # ===================================================
    # 5. GRAVAR (Com overwriteSchema porque mudou o tipo de dados)
    # ===================================================
    path_gold_filmes = "hdfs://hdfs-nn:9000/warehouse/gold.db/dim_filme"
    
    df_dim_filme.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(path_gold_filmes)

    print("✅ SUCESSO! Dimensão 'gold.dim_filme' atualizada para números.")
    df_dim_filme.show(5, truncate=False)

🧹 A criar IDs numéricos...
✅ SUCESSO! Dimensão 'gold.dim_filme' atualizada para números.
+--------+----------------------------+--------------------------------------+
|id_filme|title                       |genero                                |
+--------+----------------------------+--------------------------------------+
|1       |!Women Art Revolution       |Documentary, Musical & Performing Arts|
|2       |#ABtalks                    |                                      |
|3       |#Alive                      |horror, thriller, action, drama       |
|4       |#AnneFrank. Parallel Stories|drama, history, documentation         |
|5       |#FriendButMarried           |romance, comedy, drama                |
+--------+----------------------------+--------------------------------------+
only showing top 5 rows

